# LiTFiC — Download BOBSL from Oxford to your Drive

Downloads the licensed BOBSL files (incl. the **262 GB** Swin-V2 feature LMDB) into your Google Drive, with **resume** so it survives Colab session limits (just re-run the download cell to continue).

**Prerequisites (you must do these first — no one can do them for you):**
1. Sign the **BBC BOBSL Terms of Use** (linked from https://www.robots.ox.ac.uk/~vgg/data/bobsl/). BBC emails you a **personal password**.
2. On the BOBSL page, copy the exact 🔒 download **URLs** you need (they live under `https://thor.robots.ox.ac.uk/bobsl/v1.4/...`).

### ⚠️ Important architecture note
Downloading the 262 GB LMDB **to Drive is fine for storage**, but the later *subset carve* must **read** that LMDB with fast random access — which is slow/unreliable over Drive's FUSE mount, and Colab has no 262 GB local disk. **Plan:** keep the full data on Drive as the master copy, then carve on a **VM with a ≥300 GB local SSD** (pull the file from Drive to the VM's local disk first). See `docs/litfic-data-download.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## CONFIG — credentials + the files to fetch
Paste your BBC-provided password and the exact URLs from the BOBSL page. Keep only the files you need. The feature LMDB and pseudo-label LMDB are what the repo consumes; BLIP2 captions + CSLR annotations are small.

In [ ]:
import os, getpass

DEST = '/content/drive/MyDrive/bobsl'          # where files land on Drive
USER = 'bobsl'                                 # BOBSL HTTP user (confirm in your approval email)
PASSWORD = getpass.getpass('BOBSL password (from BBC email): ')

# Paste the exact 🔒 URLs from https://www.robots.ox.ac.uk/~vgg/data/bobsl/ :
URLS = [
    # 'https://thor.robots.ox.ac.uk/bobsl/v1.4/<SWIN_FEATURES_V2_LMDB>',   # ~262 GB
    # 'https://thor.robots.ox.ac.uk/bobsl/v1.4/<SWIN_V2_FEATURES_PSEUDO_LABELS_LMDB>',
    # 'https://thor.robots.ox.ac.uk/bobsl/v1.4/<BACKGROUND_CAPTIONS_BLIP2>',
]
os.makedirs(DEST, exist_ok=True)
assert URLS, 'Paste at least one URL from the BOBSL download page.'
print(f'{len(URLS)} file(s) -> {DEST}')

In [ ]:
# Resumable download. Re-run this cell after a session drop -- wget -c continues
# each partial file. --tries=0 retries transient network errors automatically.
import shlex, subprocess
for url in URLS:
    print('\n=== ', url)
    cmd = (f"wget -c --tries=0 --timeout=120 --retry-connrefused "
           f"--user={shlex.quote(USER)} --password={shlex.quote(PASSWORD)} "
           f"-P {shlex.quote(DEST)} {shlex.quote(url)}")
    subprocess.run(cmd, shell=True)
print('\nDownload pass complete (re-run to resume any partial files).')

In [ ]:
# Check what arrived and its size.
!ls -lh {DEST}
!du -sh {DEST}/* 2>/dev/null

In [ ]:
# The feature/PL downloads are LMDB archives. If a file arrives as a .tar/.zip,
# unpack it into a folder (LMDB folders contain data.mdb). Adjust the name.
# Example:
# !cd {DEST} && tar -xf <feats_lmdb>.tar   # -> folder with data.mdb
print('If your LMDB came as a folder with data.mdb already, nothing to unpack.')

## Also grab the small CSLR annotations (1.9 GB, Google Drive)
The `bobsl.zip` (vocab, info, subtitles, synonyms, splits) is linked from https://gulvarol.github.io/cslr2/data.html . Download it to Drive (via the browser or `gdown`), unzip, and note the paths — they become the `meta/` folder used by the subset notebook.

## Next
1. Full data now on Drive (master copy).
2. **Carve the subset on a ≥300 GB-local-disk VM** (not Colab): pull the LMDB from Drive to the VM's local SSD, run `scripts/subset_bobsl_lmdb.py`, upload the small subset to Kaggle. Details in `docs/litfic-data-download.md`.
3. If you prefer to try carving directly on Colab from Drive anyway, use `notebooks/colab_subset.ipynb` — but expect it to be slow, and it may fail on the FUSE mmap for the 262 GB file.